In [0]:
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io
import re

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:

url = "https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv"

response = requests.get(url, timeout=120)  # large file, longer timeout
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text), low_memory=False)

print(f"Full dataset: {df_raw.shape}")

# Filter to Scotland -- ATCOCode starts with '6'
df_scotland = df_raw[df_raw["ATCOCode"].astype(str).str.startswith("6")].copy()

print(f"Scotland only: {df_scotland.shape}")
print(df_scotland.dtypes)
df_scotland.head(5)

In [0]:
print(df_raw["StopType"].value_counts())
print()
print(df_raw["Status"].value_counts())

In [0]:
def clean_col_name(name):
    name = name.lower()
    name = re.sub(r"[^\w]", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

df_scotland.columns = [clean_col_name(c) for c in df_raw.columns]
print(df_scotland.columns.tolist())

In [0]:
df_bronze = spark.createDataFrame(df_scotland)
df_bronze.printSchema()

In [0]:
# Cell 7: Sanity check
df_bronze.show(10, truncate=False)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.bronze.transport_stops")
)

In [0]:
# Source: DfT NaPTAN - National Public Transport Access Nodes, Scotland
# URL: https://beta-naptan.dft.gov.uk/download/Scotland/csv
# Covers all public transport access points: bus stops, rail stations,
# ferry terminals, tram and metro stops.
# Coordinates in EPSG:27700 (British National Grid).
# Open Government Licence. Updated regularly by local authorities.